# LV 80 expansion cost: INFLEX vs HEMS

Single real SWF grid, full chronological year (8,760 h), dedicated EV charging
sessions, annual storage closure.

**Scope.** This compares *dispatch* on one topology: both controllers serve the
identical EV session obligations with the identical fixed assets, so only the
charging and battery schedules differ. It is **not** a real-vs-synthetic paired
comparison -- the two networks partition the same buildings along different
boundaries, so LV 80 has no synthetic counterpart. Regional cost totals need the
complete grid set; one grid cannot stand in for them.

Prerequisite (already run) from `GridExpand/5.postprocessing`:

    for spec in pre:pre:lv080_pre \
                post-inflex-heuristic:post:lv080_post_inflex \
                post-hems-heuristic:post:lv080_post_hems; do
      IFS=: read run stage key <<< "$spec"
      uv run python -m expansion.grid_expansion \
        --run-name forchheim_2045_paired_full_year_lv080_real_swf_$run \
        --data-source real_swf --stage $stage \
        --analysis-key forchheim_2045_paired_full_year_$key \
        --replace --plz 91301
    done


In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent          # GridExpand/5.postprocessing
for entry in (str(ROOT), str(ROOT.parent)):
    if entry not in sys.path:
        sys.path.insert(0, entry)

import pandas as pd
from expansion import grid_expansion as ge

pd.set_option("display.width", 200)

CASES = {
    "status quo": "forchheim_2045_paired_full_year_lv080_pre",
    "INFLEX":     "forchheim_2045_paired_full_year_lv080_post_inflex",
    "HEMS":       "forchheim_2045_paired_full_year_lv080_post_hems",
}
tables = {name: ge.load_expansion_overview(analysis_key=key) for name, key in CASES.items()}

# Fail loudly rather than silently comparing an incomplete or non-converged grid.
for name, t in tables.items():
    status = t["real_grid_status"]
    assert not t["analysis_run"].empty, f"{name}: analysis not materialized"
    assert (status["cost_status"] == "complete").all(), f"{name}: {status['cost_status'].tolist()}"
    assert (status["n_failed_timesteps"] == 0).all(), f"{name}: non-converged timesteps"
print("all three analyses present, cost-complete, fully converged")

all three analyses present, cost-complete, fully converged


## Cost

In [2]:
cost = pd.DataFrame(
    {name: t["cost_summary"].iloc[0] for name, t in tables.items()}
).T[[
    "cable_segments", "cable_segments_overloaded",
    "reinforcement_150_count", "reinforcement_185_count", "reinforcement_240_count",
    "cable_cost_eur",
    "transformers_overloaded", "transformer_cost_eur",
    "total_cost_eur",
]]
cost

,cable_segments,cable_segments_overloaded,reinforcement_150_count,reinforcement_185_count,reinforcement_240_count,cable_cost_eur,transformers_overloaded,transformer_cost_eur,total_cost_eur
status quo,532,7,7,0,0,13854.999678,0,0.0,13854.999678
INFLEX,532,38,50,2,6,86606.000319,1,100000.0,186606.000319
HEMS,532,32,42,0,1,61612.000109,1,48000.0,109612.000109


In [3]:
inflex = cost.loc["INFLEX", "total_cost_eur"]
hems = cost.loc["HEMS", "total_cost_eur"]
print(f"INFLEX   {inflex:>12,.0f} EUR")
print(f"HEMS     {hems:>12,.0f} EUR")
print(f"saving   {inflex - hems:>12,.0f} EUR   ({100 * (inflex - hems) / inflex:.1f} % lower under HEMS)")

INFLEX        186,606 EUR
HEMS          109,612 EUR
saving         76,994 EUR   (41.3 % lower under HEMS)


## Physical drivers

Expansion is triggered by the **annual maximum** against discrete cable and
transformer increments, so the tail matters far more than the average.

In [4]:
from sqlalchemy import text
from common.database import SurroGridDatabase

RUNS = {
    "status quo": ("forchheim_2045_paired_full_year_lv080_real_swf_pre", "pre"),
    "INFLEX":     ("forchheim_2045_paired_full_year_lv080_real_swf_post-inflex-heuristic", "post"),
    "HEMS":       ("forchheim_2045_paired_full_year_lv080_real_swf_post-hems-heuristic", "post"),
}
query = text('''
    SELECT s.n_timesteps, s.n_failed_timesteps,
           s.trafo_loading_p50_time_percent  AS trafo_p50,
           s.trafo_loading_p95_time_percent  AS trafo_p95,
           s.trafo_loading_max_time_percent  AS trafo_max,
           s.trafo_loading_hours_above_100   AS trafo_hours_over_100,
           s.voltage_p05_load_bus_hour_pu    AS voltage_p05,
           s.boundary_peak_t_index           AS peak_hour,
           s.boundary_24h_excess_percent     AS boundary_excess_24h,
           s.boundary_168h_excess_percent    AS boundary_excess_168h
    FROM surrogrid.real_powerflow_summary s
    JOIN surrogrid.real_powerflow_run r USING (real_powerflow_run_id)
    WHERE r.run_name = :run AND s.stage = :stage
''')
db = SurroGridDatabase()
with db.engine.connect() as conn:
    loading = pd.DataFrame({
        name: pd.read_sql(query, conn, params={"run": run, "stage": stage}).iloc[0]
        for name, (run, stage) in RUNS.items()
    }).T
loading

,n_timesteps,n_failed_timesteps,trafo_p50,trafo_p95,trafo_max,trafo_hours_over_100,voltage_p05,peak_hour,boundary_excess_24h,boundary_excess_168h
status quo,8760.0,0.0,27.467897,46.679488,60.381852,0.0,0.945988,234.0,-2.978087,-2.978087
INFLEX,8760.0,0.0,42.171507,100.043300,186.589279,439.0,0.892756,1052.0,-59.973673,-50.421451
HEMS,8760.0,0.0,38.169899,87.084257,149.621959,152.0,0.901020,967.0,-59.382185,-39.360476


### Reading the boundary columns

A negative `boundary_excess` means the highest loading in the first/last 24 h (or
168 h) sits *below* the interior maximum -- the annual peak is not being pulled to
the horizon edge. The earlier TSAM runs showed the opposite: most HEMS maxima
landed exactly on a 168-hour representative-period boundary. These columns exist
so that failure mode is visible rather than averaged away.

## Where the cable cost sits

In [5]:
by_type = tables["HEMS"]["cable_summary_by_type"]
cols = ["visible_std_type", "cable_segments", "cable_segments_requiring_expansion",
        "length_km", "estimated_cost_eur"]
by_type[by_type["cable_segments_requiring_expansion"] > 0][cols] \
    .sort_values("estimated_cost_eur", ascending=False)

,visible_std_type,cable_segments,cable_segments_requiring_expansion,length_km,estimated_cost_eur
0,NKBA 4x95sm 0.6/1kV,25,11,0.423,22439.999555
1,NKBA 4x70sm 0.6/1kV,5,5,0.220,18700.000196
2,NKBA 4x50sm 0.6/1kV,9,7,0.234,10455.000111
3,NYY-J 3x150SM 0.6/1kV,92,7,2.386,9180.000208
4,NYY-J 3x95SM 0.6/1kV,11,1,0.832,752.000036
5,NYY-J 4x50,49,1,1.021,85.000004


## Caveats

1. **n = 1.** LV 80 is the largest usable real grid (325 EVs, 211 buildings,
   1.46 GWh/a) and therefore where the effect should be clearest. It says nothing
   about the median grid. The earlier TSAM conclusion was distributional -- 38 of
   45 grids worsening, a P90 across grids -- and revisiting it needs the full set.
2. **LV 113 is larger but excluded** by run configuration, and is a known
   incomplete SWF grid.
3. **Imports may legitimately differ** between controllers. Equal EV service does
   not imply equal grid import: battery round-trip losses, PV export and
   heat-pump dispatch all move. Compare component balances, not import totals.
